# Judging Instruction Following — project starter

**Purpose:** compare an LLM judge with a deterministic measurement rule while keeping compliance distinct from overall quality. Read `datasheet.md` and `system_card.md` first.


## 1. Check the package and lock common evaluation responses

The first code cell finds this track's folder or downloads its zip from the course Box folder, checks that the download matches the course copy (SHA-256 hash), and unpacks it next to this notebook. If the course link cannot be downloaded directly, the cell tells you how to upload the zip yourself; the same check runs either way.


In [ ]:
from pathlib import Path
import hashlib
import json
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

TRACK_ZIP = {'file': 'track_4_instructions.zip', 'bytes': 1420134, 'sha256': 'a5afba761e928c807449be04e8f0e436f3d1f2adc8309309842c815c55052a62', 'url': 'https://cmu.box.com/shared/static/kv6tzes7xjac5pxaen6ydamo32hzed77.zip'}

def get_track_package(zip_info):
    """Use this track's folder if it is here; otherwise download, check, and unpack its zip."""
    name = zip_info["file"].removesuffix(".zip")
    here = Path.cwd()
    for folder in [here, here / name]:
        if folder.name == name and (folder / "metadata.csv").exists():
            return folder
    archive = here / zip_info["file"]
    upload_by_hand = (f"Download {zip_info['file']} from the course Box folder yourself, upload it to "
                      f"{here} (in Colab, use the Files panel), and rerun this cell.")
    if not archive.exists():
        if not zip_info["url"]:
            raise FileNotFoundError(upload_by_hand)
        print(f"Downloading {zip_info['file']} ({zip_info['bytes'] / 1e6:.0f} MB) ...")
        partial = here / (zip_info["file"] + ".part")
        try:
            urllib.request.urlretrieve(zip_info["url"], partial)
        except OSError as error:
            partial.unlink(missing_ok=True)
            raise FileNotFoundError(f"The download did not work ({error}). {upload_by_hand}") from None
        if hashlib.sha256(partial.read_bytes()).hexdigest() != zip_info["sha256"]:
            partial.unlink()  # often a Box sign-in page rather than the zip
            raise FileNotFoundError(f"The download is not the course zip. {upload_by_hand}")
        partial.replace(archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != zip_info["sha256"]:
        raise ValueError(f"{archive.name} does not match the course copy. Delete it and "
                         "download it from the course Box folder again.")
    with zipfile.ZipFile(archive) as z:
        z.extractall(here)
    print(f"Unpacked {archive.name} into {name}/.")
    return here / name

ROOT = get_track_package(TRACK_ZIP)

def wilson_interval(correct, n, z=1.96):
    """Wilson interval for a binomial proportion; returns (low, high)."""
    if n == 0:
        return (np.nan, np.nan)
    p = correct / n
    den = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / den
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / den
    return (center - half, center + half)

def paired_bootstrap_delta(ai_correct, baseline_correct, B=4000, seed=2027):
    """Percentile interval for mean(AI correct - baseline correct), paired by case."""
    ai = np.asarray(ai_correct, dtype=float)
    base = np.asarray(baseline_correct, dtype=float)
    if len(ai) != len(base) or len(ai) == 0:
        raise ValueError("Paired comparison requires the same nonempty cases.")
    rng = np.random.default_rng(seed)
    draws = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, len(ai), len(ai))
        draws[b] = np.mean(ai[idx] - base[idx])
    return float(np.mean(ai-base)), tuple(np.quantile(draws, [0.025, 0.975]))

def result_row(system, correct):
    correct = np.asarray(correct, dtype=bool)
    lo, hi = wilson_interval(int(correct.sum()), len(correct))
    return {
        "system": system,
        "correct": int(correct.sum()),
        "n": int(len(correct)),
        "estimate": correct.mean(),
        "ci_low": lo,
        "ci_high": hi,
    }

def select_cases(cases, n, id_col, confidence_col, seed=2027):
    """Lock the selection rule before anything is displayed.

    Among cases where the baseline and AI differ in correctness, draw up to n/2
    where the AI is wrong and n/2 where the baseline is wrong (seeded draw).
    """
    pool = cases.loc[cases["baseline_correct"] != cases["ai_correct"]]
    if pool.empty:
        pool = cases
    parts = [group.sample(min(len(group), n // 2), random_state=seed)
             for _, group in pool.groupby("ai_correct")]
    return pd.concat(parts).sort_values(["ai_correct", confidence_col, id_col], ascending=[True, False, True])

print("Package folder:", ROOT.name)


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

meta = pd.read_csv(ROOT / "metadata.csv")
responses = pd.read_json(ROOT / "data" / "responses.jsonl", lines=True)
rules = pd.read_csv(ROOT / "ai_outputs" / "rule_checks.csv")
judge = pd.read_json(ROOT / "ai_outputs" / "judge_outputs.jsonl", lines=True)
assert meta.response_id.is_unique and responses.response_id.is_unique
assert set(meta.response_id) == set(responses.response_id)

rule_unit = rules.groupby("response_id", as_index=False).agg(
    rule_pass=("rule_pass", "all"), n_constraints=("constraint_id", "nunique")
)
cases = (meta.merge(responses[["response_id","prompt","response","length_tokens","hit_token_limit"]], on="response_id", validate="one_to_one")
             .merge(rule_unit, on="response_id", validate="one_to_one"))
# One fixed training/evaluation split (a quarter of the responses for evaluation).
# The two responses to a prompt share its instructions, so each prompt stays on one side.
strata = cases.constraint_family.astype(str) + "|" + cases.rule_pass.astype(str)
folds = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=2027)
train_rows, eval_rows = next(folds.split(cases, strata, groups=cases.prompt_id))
assert not set(cases.prompt_id.iloc[train_rows]) & set(cases.prompt_id.iloc[eval_rows])
train_ids, eval_ids = list(cases.response_id.iloc[train_rows]), list(cases.response_id.iloc[eval_rows])
print(f"Responses: {len(cases):,}; training responses: {len(train_ids):,}; evaluation responses: {len(eval_ids):,}")


## 2. Fit the statistical baseline and compare it fairly

The fitted logistic baseline uses response length, constraint family, and source-system indicator. The family-specific majority floor is learned on training responses. Parse failures count as failed end-to-end judge measurements.


In [ ]:
D = cases.set_index("response_id")
train = D.loc[train_ids]
evaluation = D.loc[eval_ids]
features = ["length_tokens", "constraint_family", "model"]

family_majority = train.groupby("constraint_family").rule_pass.mean().ge(.5)
global_majority = bool(train.rule_pass.mean() >= .5)
floor_pred = evaluation.constraint_family.map(family_majority).fillna(global_majority).astype(bool).to_numpy()

prep = ColumnTransformer([
    ("length", StandardScaler(), ["length_tokens"]),
    ("categories", OneHotEncoder(handle_unknown="ignore"), ["constraint_family", "model"]),
])
baseline = make_pipeline(prep, LogisticRegression(max_iter=1000))
baseline.fit(train[features], train.rule_pass.astype(int))
baseline_pred = baseline.predict(evaluation[features]).astype(bool)

jv1 = judge.loc[judge.prompt_version == "v1"].drop_duplicates("response_id").set_index("response_id")
judge_raw = jv1.reindex(eval_ids).judge_pass
judge_pred = judge_raw.fillna(False).astype(bool).to_numpy()
parse_failed = jv1.reindex(eval_ids).parse_error.fillna(True).astype(bool).to_numpy()
truth = evaluation.rule_pass.astype(bool).to_numpy()

floor_correct = floor_pred == truth
baseline_correct = baseline_pred == truth
ai_correct = (judge_pred == truth) & (~parse_failed)
results = pd.DataFrame([
    result_row("family-majority floor", floor_correct),
    result_row("logistic response-feature baseline", baseline_correct),
    result_row("cached LLM judge (v1)", ai_correct),
])
display(results.style.format({"estimate":"{:.3f}", "ci_low":"{:.3f}", "ci_high":"{:.3f}"}))
delta, delta_ci = paired_bootstrap_delta(ai_correct, baseline_correct)
print(f"Paired judge - baseline accuracy: {delta:+.3f} "
      f"(95% interval {delta_ci[0]:+.3f}, {delta_ci[1]:+.3f}; n={len(truth)})")


## 3. Check cached judge measurements and rule coverage


In [ ]:
checks = pd.Series({
    "responses": len(meta),
    "rule-check records": len(rules),
    "responses with no rule check": int((~meta.response_id.isin(rules.response_id)).sum()),
    "responses cut off at the 640-token limit": int(responses.hit_token_limit.sum()),
    "judge records": len(judge),
    "duplicate (response, prompt version) keys": int(judge.duplicated(["response_id","prompt_version"]).sum()),
    "judge models": ", ".join(sorted(judge.judge_model.dropna().astype(str).unique())),
    "prompt versions": ", ".join(sorted(judge.prompt_version.dropna().astype(str).unique())),
    "parse failures": int(judge.parse_error.fillna(True).sum()),
    "parse-failure rate": judge.parse_error.fillna(True).mean(),
})
display(checks.to_frame("value"))
display(rules.groupby("constraint_family").agg(
    responses=("response_id","nunique"), pass_rate=("rule_pass","mean")
).round(3))


## 4. Browse selected instructions and responses

Selection rule: among common evaluation cases where the fitted baseline and judge differ in correctness, draw three where the judge is wrong and three where the baseline is wrong (seed 2027). Cases are selected before text is printed.


In [ ]:
case_table = pd.DataFrame({
    "response_id": eval_ids,
    "truth": truth,
    "baseline_pred": baseline_pred,
    "judge_pred": judge_pred,
    "baseline_correct": baseline_correct,
    "ai_correct": ai_correct,
    "judge_confidence": jv1.reindex(eval_ids).judge_confidence.fillna(-1).to_numpy(),
})
selected = select_cases(case_table, 6, "response_id", "judge_confidence")
E = evaluation
for _, row in selected.iterrows():
    rec = E.loc[row.response_id]
    print("="*88)
    print(row[["response_id","truth","baseline_pred","judge_pred","judge_confidence"]].to_dict())
    print("FAMILY:", rec.constraint_family, "| SOURCE MODEL:", rec.model,
          "| CUT OFF AT 640 TOKENS" if rec.hit_token_limit else "")
    print("PROMPT:", rec.prompt)
    print("RESPONSE:\n", rec.response[:1600])
    print("\nJUDGE RATIONALE (v1):\n", str(jv1.rationale.get(row.response_id, "(no v1 record)"))[:1200])


**Case note:** Is the rule reference capturing the client-relevant construct, or only formal compliance? Where does the judge add information, and where does it invent subjectivity?

> Write 3–5 sentences here.


## 5. Who is your client? Reweight to the client's mix

Lecture 9A showed that a benchmark's accuracy is a weighted average of the accuracies within groups, weighted by the benchmark's own mix of groups. Your client's mix is probably different. This section finds the AI system's accuracy in each group, then reweights it to a mix you choose for your client.

It uses **all cases**, because the AI system was not fitted on any of them. The comparison with the baseline in section 2 uses only the evaluation cases, so the overall numbers can differ a little.


In [ ]:
def accuracy_by_group(groups, correct):
    """One row per group: cases, the benchmark's share, and accuracy with a Wilson interval."""
    table = pd.DataFrame({"group": groups, "correct": correct})
    total = len(table)
    rows = []
    for group_name, cases in table.groupby("group"):
        n = len(cases)                       # cases in this group
        right = int(cases["correct"].sum())  # cases the AI system got right
        low, high = wilson_interval(right, n)
        rows.append({"group": group_name, "cases": n, "benchmark_share": n / total,
                     "accuracy": right / n, "ci_low": low, "ci_high": high})
    return pd.DataFrame(rows).sort_values("accuracy").reset_index(drop=True)


def plot_by_group(by_group, label):
    """Accuracy in each group with its 95% Wilson interval, lowest first."""
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(by_group) + 1.2))
    for i, row in by_group.iterrows():
        ax.errorbar(row["accuracy"], i,
                    xerr=[[row["accuracy"] - row["ci_low"]], [row["ci_high"] - row["accuracy"]]],
                    fmt="o", color="#155a8a", capsize=2)
    ax.set_yticks(range(len(by_group)))
    ax.set_yticklabels(by_group["group"])
    ax.set_xlim(0, 1)
    ax.set_xlabel(f"AI system's accuracy within each {label}, with 95% Wilson interval")
    plt.tight_layout()
    plt.show()


def reweight(by_group, shares):
    """Accuracy for a mix of groups, its standard error, and Kish's effective sample size."""
    total_share = sum(shares.values())
    estimate = 0.0
    variance = 0.0
    weight_sum = 0.0
    for _, row in by_group.iterrows():
        share = shares.get(row["group"], 0.0) / total_share  # shares are rescaled to add to 1
        p = row["accuracy"]
        n = row["cases"]
        estimate = estimate + share * p
        variance = variance + share**2 * p * (1 - p) / n
        weight_sum = weight_sum + share**2 / n
    return estimate, np.sqrt(variance), 1 / weight_sum


In [ ]:
# Groups: the type of instruction (constraint family).
# The judge (prompt v1) is right when its pass/fail call matches the rule check.
# A parse failure counts as wrong, as in section 2.
case_ids = cases["response_id"].to_numpy()
judge_v1 = jv1.reindex(case_ids)
judge_pass = judge_v1["judge_pass"].fillna(False).astype(bool).to_numpy()
parse_failed = judge_v1["parse_error"].fillna(True).astype(bool).to_numpy()
all_truth = cases["rule_pass"].astype(bool).to_numpy()
groups = cases["constraint_family"].to_numpy()
all_correct = (judge_pass == all_truth) & (~parse_failed)
GROUP_LABEL = 'type of instruction'

by_group = accuracy_by_group(groups, all_correct)
display(by_group.style.format({"benchmark_share": "{:.3f}", "accuracy": "{:.3f}",
                               "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))
plot_by_group(by_group, GROUP_LABEL)


**Reading the table.** Grouping by type of instruction is grouping by the type of case, like color and infrared photos in Lecture 9A. Each group has only about 40 responses, so the intervals are wide. Each prompt has two responses (`prompt_id`), so the standard error here is too small; Week 4's cluster bootstrap, resampling prompts, fixes that.


In [ ]:
# The benchmark's own mix of groups
benchmark_shares = {}
for _, row in by_group.iterrows():
    benchmark_shares[row["group"]] = row["benchmark_share"]

# Your client's mix: start from a copy of the benchmark's mix, then change it below.
client_shares = dict(benchmark_shares)

# ---- Edit here. Shares are rescaled to add to 1, so counts work too. ----
# client_shares["name of a group"] = 0.40
# A group that is not in your client's population can be set to 0.

benchmark_estimate, benchmark_se, benchmark_neff = reweight(by_group, benchmark_shares)
client_estimate, client_se, client_neff = reweight(by_group, client_shares)

print(f"Benchmark's mix: {benchmark_estimate:.3f} "
      f"(95% interval {benchmark_estimate - 1.96 * benchmark_se:.3f} to "
      f"{benchmark_estimate + 1.96 * benchmark_se:.3f}), from {by_group['cases'].sum():,} cases")
print(f"Your client's mix: {client_estimate:.3f} "
      f"(95% interval {client_estimate - 1.96 * client_se:.3f} to "
      f"{client_estimate + 1.96 * client_se:.3f}), worth about {client_neff:.0f} equally weighted cases")
print(f"Client minus benchmark: {client_estimate - benchmark_estimate:+.3f}")


**Where your client's mix comes from (one or two sentences).** A published figure, the client's own records, or a reasoned guess? Say which, and why it is plausible.

> Write it here.

**Population paragraph (draft it before you leave; it becomes "who the numbers are about" in Milestone 1).** In four or five sentences: name the population the benchmark describes and the one your client cares about; say how they differ, using the table above; give the reweighted accuracy with its interval and the assumption it rests on; say whether the benchmark's number is likely too high or too low for your client, or why the direction is unknown; and name the data that would check it.

> Write it here.


## 6. Project-launch handoff: start the Milestone 1 outline

Fill in what you can before you leave the launch session. The five headings below are the five parts of Milestone 1; Milestone 1 in the Project Description says what goes in each.

**The problem.**

>

**Who the numbers are about.** Start from your section 5 paragraph.

>

**EDA.** Use what you found in sections 1 to 5.

>

**Ideas for the analysis.**

>

**Open questions.**

>
